# FMCG Inventory Cost Optimization
## EOQ + MILP-Style Constrained Optimization

**Objective:** Minimize total annual inventory cost = Ordering Cost + Holding Cost  
**Method:** Economic Order Quantity (EOQ) + SLSQP Constrained Optimization  
**Dataset:** 15 FMCG products across 5 categories  
**Tools:** Python, pandas, NumPy, SciPy, Matplotlib

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from scripts.eoq_model import load_data, run_eoq, run_constrained_optimization
print('✅ Imports successful')

## 2. Load Data

In [ ]:
df = load_data('data/sample_data.csv')
print(f'Loaded {len(df)} products')
df

## 3. EOQ Theory

The **Wilson EOQ formula** minimises the sum of annual ordering and holding costs:

$$EOQ = \sqrt{\frac{2DS}{H}}$$

| Symbol | Meaning |
|--------|---------|
| D | Annual demand (units/year) |
| S | Ordering cost per order (₹) |
| H | Holding cost per unit per year (₹) |

**Total Annual Inventory Cost:**

$$TC(Q) = \frac{D}{Q} \cdot S + \frac{Q}{2} \cdot H$$

At EOQ, ordering cost = holding cost (minimum point of the U-shaped cost curve).

**Reorder Point:**

$$ROP = \frac{D}{365} \times (L + SS)$$

where L = lead time (days), SS = safety stock (days)

## 4. Compute EOQ for All Products

In [ ]:
eoq_df = run_eoq(df)
eoq_df[['product_name','eoq','annual_inv_cost','reorder_point','orders_per_year']]

## 5. Visualize EOQ Results

In [ ]:
PALETTE = ['#1B4F72', '#2E86C1', '#AED6F1', '#E74C3C', '#F39C12']
names = [n[:18]+'…' if len(n)>18 else n for n in eoq_df['product_name']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, eoq_df['eoq'], color=PALETTE[1], edgecolor='white')
ax.bar_label(bars, fmt='%.0f', padding=4, fontsize=8)
ax.set_xlabel('Optimal Order Quantity (units)')
ax.set_title('Economic Order Quantity per Product', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# EOQ Cost Curve for first product
row = eoq_df.iloc[0]
D0, S0, H0, Q0 = row['annual_demand'], row['order_cost'], row['holding_cost'], row['eoq']
q   = np.linspace(50, Q0*3, 300)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(q, (D0/q)*S0,             '--', color=PALETTE[0], lw=2, label='Ordering Cost')
ax.plot(q, (q/2)*H0,              '--', color=PALETTE[3], lw=2, label='Holding Cost')
ax.plot(q, (D0/q)*S0 + (q/2)*H0,       color=PALETTE[4], lw=3, label='Total Cost')
ax.axvline(Q0, color='#27AE60', lw=2, linestyle=':', label=f'EOQ = {Q0:.0f} units')
ax.scatter([Q0], [row['annual_inv_cost']], color='#27AE60', s=100, zorder=5)
ax.set_xlabel('Order Quantity (units)'); ax.set_ylabel('Annual Cost (₹)')
ax.set_title(f'EOQ Cost Curve — {row["product_name"]}', fontsize=13, fontweight='bold')
ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Constrained Optimization

In [ ]:
# Constraints:
#   - Max warehouse storage: 100 m² (EOQ would use ~150 m²)
#   - Capital budget: ₹5,00,000

opt_df, summary = run_constrained_optimization(
    eoq_df, budget_limit=500_000, storage_limit=100.0
)

print('=== Optimization Summary ===')
for k, v in summary.items():
    print(f'  {k:30s}: {v}')

In [ ]:
# Compare EOQ vs Constrained quantities
x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(x-w/2, eoq_df['eoq'],          w, label='EOQ (unconstrained)', color=PALETTE[1], edgecolor='white')
ax.bar(x+w/2, opt_df['opt_quantity'],  w, label='Optimised',          color=PALETTE[3], edgecolor='white', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Order Quantity (units)')
ax.set_title('EOQ vs Constrained Optimized Quantity', fontsize=13, fontweight='bold')
ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Full Results Table

In [ ]:
results = opt_df[['product_name','category','eoq','opt_quantity',
                   'annual_inv_cost','opt_annual_cost']].copy()
results['delta_cost'] = results['opt_annual_cost'] - results['annual_inv_cost']
results.columns = ['Product','Category','EOQ','Opt Qty','EOQ Cost (₹)','Opt Cost (₹)','Δ Cost (₹)']
results.style.format({'EOQ':'{:.0f}','Opt Qty':'{:.0f}',
                       'EOQ Cost (₹)':'₹{:,.0f}','Opt Cost (₹)':'₹{:,.0f}',
                       'Δ Cost (₹)':'₹{:+,.0f}'})

## 8. Key Insights

1. **EOQ minimises costs precisely** — at EOQ, ordering cost = holding cost (equal, minimum total).

2. **Storage constraints create a tradeoff** — limiting warehouse space to 100 m² (vs ~150 m² needed) forces smaller orders, meaning more ordering runs and higher total cost (~₹5,560/year extra).

3. **Soap Bar has highest inventory cost (₹8,975/yr)** due to very high demand. Negotiating lower order cost would yield the biggest savings.

4. **Toothpaste needs earliest reordering** — ROP = 247 units with a 14-day lead time requires the most proactive inventory management.

5. **Recommended policy:** Implement a continuous review (s, Q) system — place an order of size EOQ whenever stock falls to the reorder point.